In [ ]:
import copy

import torch,torchvision
from loaddata import *
from models import resnet,vit_small
import torch.nn as nn
import time
import os
from aggregate_module import *
import argparse
import torchsummary
import math
from statistics import mean
from sklearn.metrics import classification_report,precision_score,recall_score,f1_score,accuracy_score,cohen_kappa_score
from pympler import asizeof
import pdb
import pickle
import ctypes
plt.rcParams['font.sans-serif']=['SimHei'] #用来正常显示中文标签
plt.rcParams['axes.unicode_minus']=False #用来正常显示负号
import heapq
import math
import numpy as np
import random
from sklearn.metrics import pairwise_distances
from scipy.stats import pearsonr
from scipy.stats import pearsonr, spearmanr


os.environ["CUDA_VISIBLE_DEVICES"] = "1"


# Function

In [ ]:
def get_nonexistant_path(fpath):
    if not os.path.exists(fpath):
        return fpath
    # 将路径分割成根路径root和扩展名extension两部分
    root,extension=os.path.splitext(fpath)
    i=1
    new_fpath='{}_{}{}'.format(root,i,extension)
    # 是否存在相同的文件路径
    while os.path.exists(new_fpath):
        i+=1
        new_fpath='{}_{}{}'.format(root,i,extension)
    return new_fpath


class global_meters():
    def __init__(self,args):
        # 初始化对象的属性
        self.args=args
        # 全局轮次
        self.epochs=['global epochs',]
        # 每轮选择的客户端ID的信息
        self.select_clients=['selected client id in each round',]
        # 保存测试轮次中保存的全局模型
        self.model_checkpoints= {'declar':'saved global models in test round'}
        # 保存每轮训练的统计信息
        self.train_meters = {'declar':'stastic information in each training round'}
        # 保存每轮测试的统计信息
        self.test_meters = {'declar':'stastic information in each testing round'}
        # self.interval=2
        # 间隔为1
        self.interval = 1
        # 保存每个client对应的群集id信息
        self.cluster_id={'declar':'cluster id for each client,is arranged in the order of select_clients'}

    def update_train_meter(self,meter,epoch):
        # 更新训练统计信息
        if epoch%self.interval == 0:
            if epoch not in self.train_meters:
                self.train_meters[epoch]=[]
            self.train_meters[epoch].append(meter)

    def update_test_meter(self,meter,epoch):
        # 更新测试统计信息
        if epoch%self.interval==0:
            if epoch not in self.test_meters:
                self.test_meters[epoch]=[]
            self.test_meters[epoch].append(meter)

    def update_model(self,model,epoch,global_epoch):

        self.model_checkpoints[epoch] = {}
        for key in model.keys():
            # 将模型中的键和对应值，存储到字典中
            self.model_checkpoints[epoch][key] = (model[key].state_dict())



    def update(self, select_client, epoch):
        cluster_id = None
        # 更新其他信息
        if epoch%self.interval==0:
            self.select_clients.append(select_client)
            self.epochs.append(epoch)
            if cluster_id:
                new_cluster_id = {}
                for i in range(self.args.cluster_num):
                    arr = []
                    for key, value in cluster_id.items():
                        if i==value:
                            arr.append(int(key))
                    new_cluster_id[i]=sorted(arr)
                self.cluster_id[epoch]=new_cluster_id



class meters():

    def __init__(self):
        # 真实标签
        self.ground_truth = None
        # 预测标签
        self.predict = None
        # 当前client中的数据点总数
        self.num_sample =['total number of data points in this client',]
        # client中标签的id
        self.labels_id=['labels id in this client',]
        # 每个轮次的本地训练损失
        self.loss=['local training loss for every epoch',]
        # 每个本地训练批次的大小
        self.batch_size=['evey local training batch size',]
        # 当前client追踪每个类别的样本数
        self.num_per_class=['number of samples for every class in this client',]
        # 时间
        self.time=0
    def update_time(self):
        # 更新时间
        if self.time == 0:
            self.time = time.time()
        else:
            self.time = time.time()-self.time

    def update(self, ground_truth,predict,loss):
        # 更新统计信息
        ground_truth=ground_truth.cpu().numpy()
        # 预测标签
        m=torch.nn.Softmax(dim=1)
        predict=torch.argmax(m(predict),dim=1)
        predict=predict.data.cpu().numpy()
        loss=loss.detach().cpu().numpy().item()
        if self.ground_truth is None or self.predict is None:
            self.ground_truth=ground_truth
            self.predict=predict
        else:
            # 原先cat参数里面有dtype=np.int32
            self.ground_truth=np.concatenate([self.ground_truth,ground_truth],axis=0)
            self.predict=np.concatenate([self.predict,predict],axis=0)
        self.loss.append(loss)
        self.batch_size.append(len(ground_truth))
    def produce_metric(self):
        # 生成统计指标
        self.new_loss=['local training loss for every epoch',]
        self.new_loss=[los*batch for los,batch in zip(self.loss[1:],self.batch_size[1:])]
        self.new_loss=np.sum(np.array(self.new_loss))/np.sum(np.array(self.batch_size[1:]))
        self.new_loss=np.float32(self.new_loss)
        del self.loss
        self.batch_size=self.batch_size[:2]

        self.labels_id.append(sorted(list(set(self.ground_truth))))
        self.num_sample.append(len(self.ground_truth))
        self.num_per_class.append([len(self.ground_truth[self.ground_truth==label]) for label in self.labels_id[1]])
        self.meter=classification_report(self.ground_truth,self.predict,zero_division=0)
        self.precision_score=np.float32(precision_score(self.ground_truth,self.predict,average='macro',zero_division=0))
        self.recall_score = np.float32(recall_score(self.ground_truth, self.predict,average='macro',zero_division=0))
        self.f1_score = np.float32(f1_score(self.ground_truth, self.predict,average=None,zero_division=0))
        self.accuracy_score = np.float32(accuracy_score(self.ground_truth, self.predict))
        self.kappa = np.float32(cohen_kappa_score(self.ground_truth, self.predict))
        
        

        del self.ground_truth
        del self.predict
   

In [ ]:
def validate(val_loader, model):
    """
    Run evaluation
    """
    meter = meters()
    # switch to evaluate mode
    model.cuda().eval()
    # 记时
    meter.update_time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
            target = target.long().cuda()
            input_var = input.data.float().cuda()
            target_var = target.cuda()

            output = model(input_var)
            loss = nn.CrossEntropyLoss().cuda()(output, target_var)

            output = output.float().cuda()
            loss = loss.float().cuda()

            meter.update(target, output.data, loss)
      
    meter.produce_metric()
    meter.update_time()
    return meter

In [ ]:
def print_labels(epoch,new_labels):
    if new_labels is not None:
        labels=[]
        for i in range(len(new_labels)):
            labels.append(new_labels[str(i)])
        print('epoch:',epoch)
        print('clusters:', labels)

In [ ]:
def yu_train(train_loader, FL):
    """
    进行预训练，得到 m_n-1' ,M_n, m_n-1'
    """
    meter = meters()
    model = FL['model']
    model.cuda().train()

    # 定义冻结中间层的轮数
    freeze_epochs = 3
    for epoch in range(freeze_epochs):
        if epoch == freeze_epochs - 1:
            meter.update_time()
        # 冻结所有层
        for param in model.parameters():
            param.requires_grad = False
        # 解冻输入层和输出层的参数
        for param in model.conv1.parameters():
            param.requires_grad = True
        # 解冻最后一层
        for name, param in model.named_parameters():
            if name in ['linear.weight', 'linear.bias']:
                param.requires_grad = True

        for i, (input, target) in enumerate(train_loader):
            # 记录数据加载时间
            target = target.long().cuda()
            input_var = input.data.float().cuda()
            target_var = target.cuda()
            # 计算输出
            output = model(input_var)
            loss = nn.CrossEntropyLoss().cuda()(output, target_var)

            FL['optimizer'].zero_grad()
            loss.backward()
            FL['optimizer'].step()

            output = output.float()
            loss = loss.float()
            if epoch == freeze_epochs - 1:
                meter.update(target, output.data, loss)
        for param in model.parameters():
            param.requires_grad = True
    # 调整学习率
    FL['lr_scheduler'].step()
    meter.produce_metric()
    meter.update_time()

    return meter, FL

In [ ]:
def on_task_update(train_loader, FL, task_id, epoch_global, fisher_dict, optpar_dict):
    FL['model'].cuda().train()
    FL['optimizer'].zero_grad()

    for j, (input, target) in enumerate(train_loader):

        target = target.long().cuda()
        input_var = input.data.float().cuda()
        target_var = target.cuda()

        FL['optimizer'].zero_grad()

        output = FL['model'](input_var)
        loss = nn.CrossEntropyLoss().cuda()(output, target_var)

        loss.backward()

    optpar_dict['user' + str(task_id)] = {}
    fisher_dict['user' + str(task_id)] = {}

    for name, param in FL['model'].named_parameters():

        optpar_dict['user' + str(task_id)][name] = param.data.clone()

        fisher_dict['user' + str(task_id)][name] = param.grad.data.clone().pow(2)

    return fisher_dict, optpar_dict

In [ ]:
def train_ewc(train_loader, FL, local_epoch, task_id, global_epoch, fisher_dict, optpar_dict):  
    
    if global_epoch < 21:
        e_lambda = 0.75
    else:
        e_lambda = 0.80
    meter = meters()

    model = FL['model']
    model.cuda().train()

    freeze_epochs = 3
    if global_epoch == 0:
        for epoch in range(local_epoch):
            if epoch == local_epoch - 1:
                meter.update_time()
            for i, (input, target) in enumerate(train_loader):
                target = target.long().cuda()
                input_var = input.data.float().cuda()
                target_var = target
                       # compute gradient and do SGD step
                FL['optimizer'].zero_grad()

                output = model(input_var)
                loss = nn.CrossEntropyLoss().cuda()(output, target_var)


                loss.backward()
                FL['optimizer'].step()

                output = output.float()
                loss = loss.float()
                if epoch == local_epoch - 1:
                    meter.update(target, output.data, loss)
    else:
        for epoch in range(local_epoch + freeze_epochs):
            if epoch == local_epoch + freeze_epochs - 1:
                meter.update_time()

            # 只在前 freeze_epochs 轮更新输入层和分类层
            if epoch < freeze_epochs:
                # 冻结所有层
                for param in model.parameters():
                    param.requires_grad = False
                # 解冻输入层和输出层的参数
                for param in model.conv1.parameters():
                    param.requires_grad = True
                # 解冻最后一层
                for name, param in model.named_parameters():
                    if name in ['linear.weight', 'linear.bias']:
                        param.requires_grad = True

            else:
                # 设置所有参数需要梯度
                for param in model.parameters():
                    param.requires_grad = True

            for i, (input, target) in enumerate(train_loader):
                target = target.long().cuda()
                input_var = input.data.float().cuda()
                target_var = target
            
                FL['optimizer'].zero_grad()

                output = FL['model'](input_var)
                loss = nn.CrossEntropyLoss().cuda()(output, target_var)


                for name, param in FL['model'].named_parameters():
                    fisher = fisher_dict['user' + str(task_id)][name]  
                    optpar = optpar_dict['user' + str(task_id)][name]  
    
                    loss += (fisher * (optpar - param).pow(2)).sum() * e_lambda

                loss.backward()
                FL['optimizer'].step()

                output = output.float()
                loss = loss.float()
                if epoch == local_epoch - 1:
                    meter.update(target, output.data, loss)
                    
    FL['lr_scheduler'].step()
    meter.produce_metric()
    meter.update_time()
    # FL['model'].cpu()
    return meter


def vertical_mean(lst):
    num_cols = len(lst[0]) 

    averages = [mean([row[col] for row in lst]) for col in range(num_cols)]

    return averages

In [ ]:
def train(train_loader, FL, local_epoch, global_epoch):
   
    meter = meters()
   
    model = FL['model']
    model.cuda().train()

    for epoch in range(local_epoch):    
        for i, (input, target) in enumerate(train_loader):
              
            target = target.long().cuda()
            input_var = input.data.float().cuda()
            target_var = target

            output = FL['model'](input_var)
            loss = nn.CrossEntropyLoss().cuda()(output, target_var)

            FL['optimizer'].zero_grad()
            loss.backward()
            FL['optimizer'].step()
        
            output = output.float()
            loss = loss.float()
            if epoch == local_epoch - 1:
                meter.update(target,output.data,loss)

    FL['lr_scheduler'].step()
    meter.produce_metric()
    meter.update_time()

    return meter

In [ ]:
# 中间层拉直
def yiwei2(FL_dict, num_client):
    params_tensor = [None] * num_client
    
    for i in range(num_client):
        model = FL_dict['user'+str(i)]['model']  # 获取模型
        params_list = []
        
        for param_name, param_tensor in model.named_parameters():
            if not param_name.startswith('conv1') and not param_name.startswith('linear'):
                params_list.append(param_tensor.view(-1))
        
        params_tensor[i] = torch.cat(params_list, dim=0)
    
    return params_tensor

In [ ]:
def xiangsi(params_tensor, params_tensor2):
    

    array1 = params_tensor.detach().cpu().numpy()
    array2 = params_tensor2.detach().cpu().numpy()
    
   
    
    # 计算欧氏距离
    euclidean_distance = pairwise_distances(array1.reshape(1, -1), array2.reshape(1, -1), metric='euclidean')

    # # 计算曼哈顿距离
    # manhattan_distance = pairwise_distances(array1.reshape(1, -1), array2.reshape(1, -1), metric='manhattan')

    # # 计算余弦距离
    # cosine_distance = pairwise_distances(array1.reshape(1, -1), array2.reshape(1, -1), metric='cosine')

    # # 计算切比雪夫距离
    # chebyshev_distance = pairwise_distances(array1.reshape(1, -1), array2.reshape(1, -1), metric='chebyshev')

    # # 计算 Minkowski 距离
    # minkowski_distance = pairwise_distances(array1.reshape(1, -1), array2.reshape(1, -1), metric='minkowski')

    # # 计算 Pearson 相关系数
    # pearson_coefficient, _ = pearsonr(array1, array2)

    # # 计算 Spearman 相关系数
    # spearman_coefficient, _ = spearmanr(array1, array2)
    
    
    return euclidean_distance[0][0]

# 参数设置

In [ ]:
import argparse

def parse_arguments(lr=0.1, data_name='paviaU', num_spectrals=103, patch_size=9, w_slice_num=2, 
                    h_slice_num=3, test_size=0.90, num_client=6, momentum=0.9, weight_decay=1e-4, 
                    aggregate_mode='avg', select_rate=1, num_class_per_cluster=[9], cluster_num=6, dirichlet_dis=9, 
                    local_epoch=2, global_epoch=100, warm_up=2, model='resnet', 
                    data_dir='F:/FedHIC',file_name = 'gaiavg_ewc_10x10_e5054_0.15_nonwarmup'):
    parser = argparse.ArgumentParser(description='fedavg primary testing')
    parser.add_argument('--lr', default=lr, type=float, help='learning rate')
    parser.add_argument('--data-name', default=data_name, type=str, help='choose the data')
    parser.add_argument('--num-spectrals', default=num_spectrals, type=int, help='')
    parser.add_argument('--patch-size', default=patch_size, type=int, help='')
    parser.add_argument('--w-slice-num', default=w_slice_num, type=int, help='')
    parser.add_argument('--h-slice-num', default=h_slice_num, type=int, help='')
    parser.add_argument('--test-size', default=test_size, type=float, help='')
    parser.add_argument('--num-client', default=num_client, type=int, help='choose the number of clients')
    parser.add_argument('--momentum', default=momentum, type=float, help='')
    parser.add_argument('--weight-decay', default=weight_decay, type=float, help='')
    parser.add_argument('--aggregate-mode', default=aggregate_mode, type=str, help='')
    parser.add_argument('--select-rate', default=select_rate, type=float, help='')
    parser.add_argument('--num-class-per-cluster', default=num_class_per_cluster, type=list, help='choose the number of class per cluster in dataset')
    parser.add_argument('--cluster-num', default=cluster_num, type=int, help='choose the number of clusters to aggregate')
    parser.add_argument('--dirichlet-dis', default=dirichlet_dis, type=int, help='choose the paramter of dirichlet distribution of data')
    parser.add_argument('--local-epoch', default=local_epoch, type=int, help='number of local epoch')
    parser.add_argument('--global-epoch', default=global_epoch, type=int, help='number of global epoch')
    parser.add_argument('--warm-up', default=warm_up, type=int, help='number of fedavg before clustering')
    parser.add_argument('--model', default=model, type=str, help='choose the deep learning model to run')
    parser.add_argument('--data-dir', default=data_dir, type=str, help='output: number of local epoch')
    parser.add_argument('--file-name', default=file_name, type=str, help='filename: head of filename')
    
    args, unknown = parser.parse_known_args()
    return args

In [ ]:
# 创建一个命名空间对象，并手动添加参数作为属性
args = argparse.Namespace()
args.lr=0.1
args.data_name='paviaU'
args.num_spectrals=103
args.patch_size=9
args.w_slice_num=2
args.h_slice_num=3
args.test_size=0.95
args.num_client=3
args.momentum=0.9
args.weight_decay=1e-4,
args.aggregate_mode='avg'
args.select_rate=1
args.num_class_per_cluster=[9]
args.cluster_num=6
args.dirichlet_dis=9
args.local_epoch=2
args.global_epoch=80
args.warm_up=2
args.model='resnet'
args.data_dir='Results/open-3/seed-42/'
args.file_name = 'Results/open-3/seed-42/'

In [ ]:
# 调用函数进行参数解析
args = parse_arguments(lr=args.lr, data_name=args.data_name, num_spectrals=args.num_spectrals, patch_size=args.patch_size, 
                       w_slice_num=args.w_slice_num, h_slice_num=args.h_slice_num, test_size=args.test_size, 
                       num_client=args.num_client, momentum=args.momentum, weight_decay=args.weight_decay, 
                       aggregate_mode=args.aggregate_mode, select_rate=args.select_rate, num_class_per_cluster=args.num_class_per_cluster, 
                       cluster_num=args.cluster_num, dirichlet_dis=args.dirichlet_dis, local_epoch=args.local_epoch, 
                       global_epoch=args.global_epoch, warm_up=args.warm_up, model=args.model, data_dir=args.data_dir,file_name = args.file_name )

In [ ]:
args

# 分数据

# New_version with hard_data

In [ ]:
def get_data():
    # dataset = ['paviaU_hard_open', 'pavia', '1Urban', 'Houston_hard_open']
    # dataset = ['paviaU_hard_closed', 'pavia', '1Urban', 'Houston_hard_closed']
    dataset = ['paviaU_hard_open', 'Houston_hard_open', 'IndianPine']

    dataset_img = [0 for i in range(len(dataset))]

    TR_gt = [0 for i in range(len(dataset))]
    TE_gt = [0 for i in range(len(dataset))]
    dataset_gt = [0 for i in range(len(dataset))]

    path = '/media/HSI-Federated Learning/data/'


    # closed-set
    # for i in range(len(dataset)):
    #     data = scio.loadmat(path + dataset[i] + '.mat')
    #     if dataset[i] in ['paviaU_hard_closed', 'Houston_hard_closed']:
    #         dataset_img[i] = data['input']
    #         TR_gt[i] = data['TR']
    #         TE_gt[i] = data['TE']
    #     elif dataset[i] == 'pavia':
    #         dataset_img[i] = data[dataset[i]]
    #         dataset_gt[i] = scio.loadmat(path + dataset[i] + '_gt_4.mat')['map']
    #     else:
    #         dataset_img[i] = data['Urban']
    #         dataset_gt[i] = scio.loadmat(path + dataset[i] + '_gt_4.mat')['map']


    # open-set
    # for i in range(len(dataset)):
    #     data = scio.loadmat(path + dataset[i] + '.mat')
    #     if dataset[i] in ['paviaU_hard_open', 'Houston_hard_open']:
    #         dataset_img[i] = data['input']
    #         TR_gt[i] = data['TR']
    #         TE_gt[i] = data['TE']
    #     elif dataset[i] == 'pavia':
    #         dataset_img[i] = data[dataset[i]]
    #         dataset_gt[i] = scio.loadmat(path + dataset[i] + '_gt.mat')['pavia_gt']
    #     else:
    #         dataset_img[i] = data['Urban']
    #         dataset_gt[i] = scio.loadmat(path + dataset[i] + '_gt_4.mat')['map']

    for i in range(len(dataset)):
        data = scio.loadmat(path + dataset[i] + '.mat')
        dataset_img[i] = data['input']
        TR_gt[i] = data['TR']
        TE_gt[i] = data['TE']


    return dataset, dataset_img, dataset_gt, TR_gt, TE_gt

In [ ]:
seed = 42

random.seed(seed)
os.environ['PYTHONHASHSEED'] =str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic =True

In [ ]:
# 加载数据集，分成dataset_img 和 dataset_gt
dataset, dataset_img, dataset_gt, TR_gt, TE_gt = get_data()

dataset_pu = [dataset_img[0].shape[2], dataset_img[1].shape[2], dataset_img[2].shape[2]]

split = 20
clients = {}
# train_shape = []
for i in range(len(dataset)):
    clients['client' + str(i)] = {}
    clients['client' + str(i)]['train'] = {}
    clients['client' + str(i)]['test'] = {}
    clients['client' + str(i)]['num_class'] = {}
clients = OrderedDict(clients)

for i in range(len(dataset)):
    print("i : ", i)
    clients['client' + str(i)]['train'], clients['client' + str(i)]['test'], clients['client' + str(i)]['num_class'] = preprocess(dataset_img[i], dataset_gt[i], TR_gt[i], TE_gt[i], args.test_size, split, i)
print("各个client数据集划分完毕")

In [ ]:
clients['client' + str(0)]['train'][0].shape

In [ ]:
clients['client' + str(0)]['test'][0].shape

# 分模型

In [ ]:
if args.model=='resnet':
    model = resnet.ResNet18(num_spectrals=dataset_pu[0], num_class=clients['client' + str(0)]['num_class'])

elif args.model=='vit':
    model = vit_small.ViT(
            image_size = clients['client' + str(0)]['train'][0].shape[1],
            patch_size = 4,
            num_classes = max(clients['client' + str(i)]['num_class']),
            dim = int(512),
            depth = 4,
            heads = 6,
            mlp_dim = 256,
            dropout = 0.1,
            emb_dropout = 0.1
            )

In [ ]:
def get_model_memory_usage(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad) * 4
b = get_model_memory_usage(model) / (1024 ** 2)

In [ ]:
global_model={}
global_model['0']=copy.deepcopy(model)
num_client_per_round=int(args.select_rate*args.num_client)

In [ ]:
global_meter=global_meters(args)

In [ ]:
global_meter2=global_meters(args)

In [ ]:
global_meter3=global_meters(args)

In [ ]:
global_meterM2=global_meters(args)

In [ ]:
FL_dict= {}
for i in range(args.num_client):
    FL_dict['user'+str(i)]={}
    FL_dict['user'+str(i)]['model'] = resnet.ResNet18(num_spectrals=dataset_pu[i], num_class=clients['client' + str(i)]['num_class'])
    FL_dict['user'+str(i)]['optimizer'] = torch.optim.SGD(FL_dict['user'+str(i)]['model'].parameters(), 0.1,
                                    momentum=0.9,
                                    weight_decay=1e-4)
    FL_dict['user'+str(i)]['lr_scheduler']=torch.optim.lr_scheduler.MultiStepLR(FL_dict['user'+str(i)]['optimizer'],
                                                            milestones=[50, 75], last_epoch=- 1)
FL_dict=OrderedDict(FL_dict)

# 初始化存东西的变量

In [ ]:
new_labels=None

In [ ]:
fisher_dict1 = {}
optpar_dict1 = {}

for i in range(args.num_client):
    fisher_dict1['user' + str(i)] = {}
    optpar_dict1['user' + str(i)] = {}

In [ ]:
fisher_dict1_cur = {}
optpar_dict1_cur = {}

for i in range(args.num_client):
    fisher_dict1_cur['user' + str(i)] = {}
    optpar_dict1_cur['user' + str(i)] = {}

In [ ]:
fff_cu = {}
opt_cu = {}

for i in range(args.num_client):
    fff_cu['user' + str(i)] = {}
    opt_cu['user' + str(i)] = {}

In [ ]:
# # 保存旧任务中的  FIM信息矩阵  和  优化参数
fisher_dict2 = {}
optpar_dict2 = {}

for i in range(args.num_client):
    fisher_dict2['user' + str(i)] = {}
    optpar_dict2['user' + str(i)] = {}

In [ ]:
# test1的 Acc
acc1 = [[0.0 for i in range(args.global_epoch)]]*args.num_client
acc1 = pd.DataFrame(acc1)
# test2的Acc
acc2 = [[0.0 for i in range(args.global_epoch)]]*args.num_client
acc2 = pd.DataFrame(acc2)
# test3的Acc
acc3 = [[0.0 for i in range(args.global_epoch)]]*args.num_client
acc3 = pd.DataFrame(acc3)

client_best_acc = [0.0 for _ in range(args.num_client)]


precision = [[0.0 for i in range(args.global_epoch)]] * args.num_client
precision = pd.DataFrame(precision)
## 存储各个client下的acc
recall = [[0.0 for i in range(args.global_epoch)]] * args.num_client
recall = pd.DataFrame(recall)
# 存储各个client下的acc
ffff = [[0.0 for i in range(args.global_epoch)]] * args.num_client
ffff = pd.DataFrame(ffff)

# 存最好一代的recall和f1还有precision
precision_best = [0.0 for _ in range(args.num_client)]
recall_best = [0.0 for _ in range(args.num_client)]
f1_best = [0.0 for _ in range(args.num_client)]
kappa_best = [0.0 for _ in range(args.num_client)]

In [ ]:
accM2 = [[0.0 for i in range(args.global_epoch)]]*args.num_client
accM2 = pd.DataFrame(accM2)

In [ ]:
# test1
all_jieguo1 = {}
for j in range(args.global_epoch):
    all_jieguo1['epoch' + str(j)] = {}
    for i in range(len(dataset)):
        all_jieguo1['epoch' + str(j)]['client' + str(i)] = {}
        # print('client' + str(i))
    all_jieguo1['epoch' + str(j)]['avg'] = {}
all_jieguo1 = OrderedDict(all_jieguo1)

In [ ]:
# test1
all_jieguoM2 = {}
for j in range(args.global_epoch):
    all_jieguoM2['epoch' + str(j)] = {}
    for i in range(len(dataset)):
        all_jieguoM2['epoch' + str(j)]['client' + str(i)] = {}
        # print('client' + str(i))
    all_jieguoM2['epoch' + str(j)]['avg'] = {}
all_jieguoM2 = OrderedDict(all_jieguoM2)

In [ ]:
# test2
all_jieguo2 = {}
for j in range(args.global_epoch):
    all_jieguo2['epoch' + str(j)] = {}
    for i in range(len(dataset)):
        all_jieguo2['epoch' + str(j)]['client' + str(i)] = {}
        # print('client' + str(i))
    all_jieguo2['epoch' + str(j)]['avg'] = {}
all_jieguo2 = OrderedDict(all_jieguo2)

In [ ]:
# test3
all_jieguo3 = {}
for j in range(args.global_epoch):
    all_jieguo3['epoch' + str(j)] = {}
    for i in range(len(dataset)):
        all_jieguo3['epoch' + str(j)]['client' + str(i)] = {}
        # print('client' + str(i))
    all_jieguo3['epoch' + str(j)]['avg'] = {}
all_jieguo3 = OrderedDict(all_jieguo3)

In [ ]:
# 欧氏距离 euclidean
xiangsi1 = {}
for j in range(args.global_epoch):
    xiangsi1['epoch' + str(j)] = {}
    for i in range(len(dataset)):
        xiangsi1['epoch' + str(j)]['client' + str(i)] = {}
        # print('client' + str(i))
    xiangsi1['epoch' + str(j)]['avg'] = {}
xiangsi1 = OrderedDict(xiangsi1)

In [ ]:
xs1 = [[0.0 for i in range(args.global_epoch)]]*args.num_client
xs1 = pd.DataFrame(xs1)
xs_weight = [[0.0 for i in range(args.global_epoch)]]*args.num_client
xs_weight = pd.DataFrame(xs_weight)

# dataloader

In [ ]:
data_loader = {}
# for i in range(4):
for i in range(args.num_client):
    if i in [0,1]:
        data_loader['client'+str(i)] = {}
        data_loader['client'+str(i)]['test'] = generate_dataloader('pppppp', clients['client' + str(i)]['test'], batch_size=32)
        data_loader['client'+str(i)]['train'] = generate_dataloader('pppppp', clients['client' + str(i)]['train'], batch_size=32)
    else:
        data_loader['client'+str(i)] = {}
        data_loader['client'+str(i)]['test'] = generate_dataloader('pppppp', clients['client' + str(i)]['test'], batch_size=32)
        data_loader['client'+str(i)]['train'] = generate_dataloader('pppppp', clients['client' + str(i)]['train'], batch_size=32)
data_loader = OrderedDict(data_loader)

# 正式联邦

In [ ]:
base_dir = '/media/HSI-Federated Learning/new/FedHIC/'

In [ ]:
for epoch in range(args.global_epoch):
    print("Global_Epoch: ", epoch)
    select_client = [i for i in range(args.num_client)]
    
    # 01  分发
    FL_dict = cluster_distribute(global_model, FL_dict, new_labels)
    
 
    for client_id in range(args.num_client):
       
#       02   微调
        meter = yu_train(data_loader['client'+str(client_id)]['train'], FL_dict['user' + str(client_id)])
    
    

    if epoch > 0:
        params_tensor = yiwei2(FL_dict, args.num_client)
    
    
    for client_id in range(args.num_client):        
         # 03   test M2  
        if epoch > 0 :
            if client_id == 0:
                print("test2:")            
      
            meterM2 = validate(data_loader['client'+str(client_id)]['test'], FL_dict['user' + str(int(client_id))]['model'])
            global_meterM2.update_test_meter(meterM2, epoch)

            # 保存acc
            accM2[epoch][client_id] = global_meterM2.test_meters[epoch][client_id].accuracy_score * 100
            
             # 保存best_model
            if accM2[epoch][client_id] > client_best_acc[client_id]:
                client_best_acc[client_id] = accM2[epoch][client_id]
                client_per_best = pd.DataFrame(np.array(client_best_acc))
                client_per_best.to_csv(base_dir+args.data_dir+'/client_best_acc.csv', index=False)

                precision_best[client_id] = global_meterM2.test_meters[epoch][client_id].precision_score
                recall_best[client_id] = global_meterM2.test_meters[epoch][client_id].recall_score
                f1_best[client_id] = 2 * (precision_best[client_id] * recall_best[client_id]) / (precision_best[client_id] + recall_best[client_id])
                kappa_best[client_id] = global_meterM2.test_meters[epoch][client_id].kappa
                
                precision_best = pd.DataFrame(np.array(precision_best))
                recall_best = pd.DataFrame(np.array(recall_best))
                f1_best = pd.DataFrame(np.array(f1_best))
                kappa_best = pd.DataFrame(np.array(kappa_best))

                precision_best.to_csv(base_dir+args.data_dir+'/best_precision.csv', index=False)
                recall_best.to_csv(base_dir+args.data_dir+'/best_recall.csv', index=False)
                f1_best.to_csv(base_dir+args.data_dir+'/best_f1.csv', index=False)
                kappa_best.to_csv(base_dir+args.data_dir+'/best_kappa.csv', index=False)

                filename = args.data_dir+'/'+'model_best{}.pth'.format(client_id)
                fpath = os.path.join(base_dir, filename)
                if os.path.exists(fpath):
                    os.remove(fpath)
                torch.save(FL_dict['user' + str(client_id)]['model'].state_dict(), fpath)
            
            # 保存acc
   
            all_jieguoM2['epoch' + str(epoch)]['client' + str(client_id)] = accM2[epoch][client_id]

            if client_id == args.num_client - 1:
                all_jieguoM2['epoch' + str(epoch)]['avg'] = (accM2[epoch][0] + accM2[epoch][1] + accM2[epoch][2])/3.0
            print("client{}当前准确率为：{}".format(client_id, accM2[epoch][client_id]))
            # 定义文件名
            filename = args.data_dir + '/testM2_jieguo.txt'
            fpath = os.path.join(base_dir, filename)
            # 判断文件是否存在，如果存在则删除
            if os.path.exists(fpath):
                os.remove(fpath)
            with open(base_dir + filename, 'w') as f:
                for key, value in all_jieguoM2.items():
                    f.write(f'{key}: {value}\n')
            accM2.to_csv(base_dir+ args.data_dir +'/testM2_acc.csv', index=False)    
        
        
        fisher_dict1, optpar_dict1 = on_task_update(data_loader['client'+str(client_id)]['train'], FL_dict['user' + str(client_id)], client_id, epoch, fisher_dict1, optpar_dict1)
        
        fff_cu = fisher_dict1 
        opt_cu = optpar_dict1 
        
        if epoch != 0:
            fisher_dict1 = fisher_dict1_cur
            optpar_dict1 = optpar_dict1_cur
        # 保存当前代的状态字典
        fisher_dict1_cur = fff_cu
        optpar_dict1_cur = opt_cu 
 
    
 # 进行训练 
        
#      03   test 2  
        if epoch > 0 :
            if client_id == 0:
                print("test2:")            

            meter1 = validate(data_loader['client'+str(client_id)]['test'], FL_dict['user' + str(int(client_id))]['model'])
            global_meter2.update_test_meter(meter1, epoch)

            # 保存acc
            acc2[epoch][client_id] = global_meter2.test_meters[epoch][client_id].accuracy_score * 100
            
             # 保存best_model
            if acc2[epoch][client_id] > client_best_acc[client_id]:
                client_best_acc[client_id] = acc2[epoch][client_id]
                client_per_best = pd.DataFrame(np.array(client_best_acc))
                client_per_best.to_csv(base_dir+args.data_dir+'/client_best_acc.csv', index=False)

                precision_best[client_id] = global_meter2.test_meters[epoch][client_id].precision_score
                recall_best[client_id] = global_meter2.test_meters[epoch][client_id].recall_score
                f1_best[client_id] = 2 * (precision_best[client_id] * recall_best[client_id]) / (precision_best[client_id] + recall_best[client_id])
                kappa_best[client_id] = global_meter2.test_meters[epoch][client_id].kappa
                
                precision_best = pd.DataFrame(np.array(precision_best))
                recall_best = pd.DataFrame(np.array(recall_best))
                f1_best = pd.DataFrame(np.array(f1_best))
                kappa_best = pd.DataFrame(np.array(kappa_best))
               
                precision_best.to_csv(base_dir+args.data_dir+'/best_precision.csv', index=False)
                recall_best.to_csv(base_dir+args.data_dir+'/best_recall.csv', index=False)
                f1_best.to_csv(base_dir+args.data_dir+'/best_f1.csv', index=False)
                kappa_best.to_csv(base_dir+args.data_dir+'/best_kappa.csv', index=False)

                filename = args.data_dir+'/'+'model_best{}.pth'.format(client_id)
                fpath = os.path.join(base_dir, filename)
                if os.path.exists(fpath):
                    os.remove(fpath)
                torch.save(FL_dict['user' + str(client_id)]['model'].state_dict(), fpath)
                        
            # 保存acc
            all_jieguo2['epoch' + str(epoch)]['client' + str(client_id)] = acc2[epoch][client_id]

            if client_id == args.num_client - 1:
                all_jieguo2['epoch' + str(epoch)]['avg'] = (acc2[epoch][0] + acc2[epoch][1] + acc2[epoch][2])/3.0
            print("client{}当前准确率为：{}".format(client_id, acc2[epoch][client_id]))
            # 定义文件名
            filename = args.data_dir + '/test2_jieguo.txt'
            fpath = os.path.join(base_dir, filename)
            # 判断文件是否存在，如果存在则删除
            if os.path.exists(fpath):
                os.remove(fpath)
            with open(base_dir + filename, 'w') as f:
                for key, value in all_jieguo2.items():
                    f.write(f'{key}: {value}\n')
            acc2.to_csv(base_dir+ args.data_dir +'/test2_acc.csv', index=False)    

    
    
    for client_id in range(args.num_client):    
        
        if epoch <= 5: 
        
            #  04    训练

            meter = train_ewc(data_loader['client'+str(client_id)]['train'], FL_dict['user' + str(client_id)], args.local_epoch, client_id, epoch, fisher_dict2, optpar_dict2)
                # 对网络的训练结束以后结束以后，得到fisher，为下一次提供，即：m_n , m_n , m_n
            fisher_dict2, optpar_dict2 = on_task_update(data_loader['client'+str(client_id)]['train'], FL_dict['user' + str(client_id)], client_id, epoch, fisher_dict2, optpar_dict2)

        else:
            if (acc1[epoch-1][client_id]/acc3[epoch-1][client_id]) < 0.88:
                meter = train_ewc(data_loader['client'+str(client_id)]['train'], FL_dict['user' + str(client_id)], args.local_epoch, client_id, epoch, fisher_dict1, optpar_dict1)
                fisher_dict2, optpar_dict2 = on_task_update(data_loader['client'+str(client_id)]['train'], FL_dict['user' + str(client_id)], client_id, epoch, fisher_dict2, optpar_dict2)
                
            else:
                meter = train_ewc(data_loader['client'+str(client_id)]['train'], FL_dict['user' + str(client_id)], args.local_epoch, client_id, epoch, fisher_dict2, optpar_dict2)
                fisher_dict2, optpar_dict2 = on_task_update(data_loader['client'+str(client_id)]['train'], FL_dict['user' + str(client_id)], client_id, epoch, fisher_dict2, optpar_dict2)
            
            
        # 定义文件名
        filename = args.data_dir+'/'+'model_state_dict{}.pth'.format(client_id)
        fpath = os.path.join(base_dir, filename)
        # 判断文件是否存在，如果存在则删除
        if os.path.exists(fpath):
            os.remove(fpath)
        # 保存新文件
        torch.save(FL_dict['user' + str(client_id)]['model'].state_dict(), fpath)

        global_meter.update_train_meter(meter, epoch)

        if client_id == 0:
            print("test3:")
        meter2 = validate(data_loader['client'+str(client_id)]['test'], FL_dict['user' + str(int(client_id))]['model'])
        global_meter3.update_test_meter(meter2, epoch)

        # 保存acc
        acc3[epoch][client_id] = global_meter3.test_meters[epoch][client_id].accuracy_score * 100

         # 存的best_model
        if acc3[epoch][client_id] > client_best_acc[client_id]:
            client_best_acc[client_id] = acc3[epoch][client_id]
            client_per_best = pd.DataFrame(np.array(client_best_acc))
            client_per_best.to_csv(base_dir+args.data_dir+'/client_best_acc.csv', index=False)

            precision_best[client_id] = global_meter3.test_meters[epoch][client_id].precision_score
            recall_best[client_id] = global_meter3.test_meters[epoch][client_id].recall_score
            f1_best[client_id] = 2 * (precision_best[client_id] * recall_best[client_id]) / (precision_best[client_id] + recall_best[client_id])
            kappa_best[client_id] = global_meter3.test_meters[epoch][client_id].kappa

            precision_best = pd.DataFrame(np.array(precision_best))
            recall_best = pd.DataFrame(np.array(recall_best))
            f1_best = pd.DataFrame(np.array(f1_best))
            kappa_best = pd.DataFrame(np.array(kappa_best))
            
            precision_best.to_csv(base_dir+args.data_dir+'/best_precision.csv', index=False)
            recall_best.to_csv(base_dir+args.data_dir+'/best_recall.csv', index=False)
            f1_best.to_csv(base_dir+args.data_dir+'/best_f1.csv', index=False)
            kappa_best.to_csv(base_dir+args.data_dir+'/best_kappa.csv', index=False)
            
            filename = args.data_dir+'/'+'model_best{}.pth'.format(client_id)
            fpath = os.path.join(base_dir, filename)
            if os.path.exists(fpath):
                os.remove(fpath)
            torch.save(FL_dict['user' + str(client_id)]['model'].state_dict(), fpath)

        
        # 保存acc

        all_jieguo3['epoch' + str(epoch)]['client' + str(client_id)] = acc3[epoch][client_id]

        if client_id == args.num_client - 1:
            all_jieguo3['epoch' + str(epoch)]['avg'] = (acc3[epoch][0] + acc3[epoch][1] + acc3[epoch][2])/3.0
        print("client{}当前准确率为：{}".format(client_id, acc3[epoch][client_id]))
        # 定义文件名
        filename = args.data_dir + '/test3_jieguo.txt'
        fpath = os.path.join(base_dir, filename)
        # 判断文件是否存在，如果存在则删除
        if os.path.exists(fpath):
            os.remove(fpath)
        with open(base_dir + filename, 'w') as f:
            for key, value in all_jieguo3.items():
                f.write(f'{key}: {value}\n')
        acc3.to_csv(base_dir+ args.data_dir +'/test3_acc.csv', index=False)    

        
#   计算相似性
    # fisher2拉成线
    if epoch > 0:
        params_tensor2 = yiwei2(FL_dict, args.num_client)
        for client_id in range(args.num_client):
    #         计算相似性
            xs1[epoch][client_id] = xiangsi(params_tensor[client_id], params_tensor2[client_id])
                
            xiangsi1['epoch' + str(epoch)]['client' + str(client_id)] = xs1[epoch][client_id]
           
            if client_id == args.num_client - 1:
                xiangsi1['epoch' + str(epoch)]['avg'] = (xs1[epoch][0] + xs1[epoch][1] + xs1[epoch][2])/3.0
           
                            
            # 定义文件名
            filename = args.data_dir + '/xiangsi1.txt'
            fpath = os.path.join(base_dir, filename)
            # 判断文件是否存在，如果存在则删除
            if os.path.exists(fpath):
                os.remove(fpath)
            with open(base_dir + filename, 'w') as f:
                for key, value in xiangsi1.items():
                    f.write(f'{key}: {value}\n')
               
            xs1.to_csv(base_dir+ args.data_dir +'/xiangsi1.csv', index=False)    
     
        arr = np.array(xs1[epoch].array)
        if all(1 < val for val in arr):
            sim = 1 / (arr + 1)
            total = sim.sum()
            v_mean = sim / total
            xs_weight[epoch]= v_mean           
        elif any(0.1< val <1 for val in arr) and np.max(arr) > 1:
            sim = np.exp(-arr/6)
            total = sim.sum()
            v_mean = sim/total
            xs_weight[epoch]= v_mean
        elif all(0.1 < val < 1 for val in arr):
            sim = 1 / np.log(arr+1.5) # -10 里面为 1    1.5
            total = sim.sum()
            v_mean = sim / total
            xs_weight[epoch]= v_mean
        else:
            sim3 = []
            map_val = 0
            for i in arr:
                if 0.1 < i < 1:
                    map_val = np.exp(-i / 10)
                elif 0.01 < i < 0.1:
                    map_val = np.exp(-i / 100)
                elif 0.001 < i < 0.01:
                    map_val = np.exp(-i / 1000)
                else:
                    pass
                sim3.append(map_val)
            sim3 = np.array(sim3)
            total = sim3.sum()
            v_mean = sim3 / total
            xs_weight[epoch]= v_mean
        
        
        xs_weight.to_csv(base_dir+ args.data_dir +'/xs_weight.csv', index=False) 
        
        
        del params_tensor
        del params_tensor2
        
   
    
#       05    聚合
    # if epoch == 0:
    new_params, weights = cluster_aggregate(args.aggregate_mode, args.warm_up, epoch, FL_dict,
                                   [len(client['train'][0]) for client in [clients['client' + str(key)] for key in select_client]], xs_weight,
                               0, args.num_client, select_client, args.cluster_num)
    weights = pd.DataFrame(weights)
    weights.to_csv(base_dir+args.data_dir+'/weights_on_data_rate.csv', index=False)

        
    if epoch == 0:
        global_model['0'] = copy.deepcopy(FL_dict['user0']['model'])
    global_model = cluster_update(global_model, new_params)

    global_meter.update(select_client, epoch)
    b = asizeof.asizeof(meter)
    print_labels(epoch, new_labels)
    a=asizeof.asizeof(global_meter)
    print('size of global meter:',round(a/(1024**2),2),'MB')

    if epoch%1==0:
#         06    分发
        FL_dict = cluster_distribute(global_model, FL_dict, new_labels)
        print("test1:")
        for client_id in range(args.num_client):
#         07    test 1   

            meter = validate(data_loader['client'+str(client_id)]['test'], FL_dict['user' + str(int(client_id))]['model'])
            global_meter.update_test_meter(meter, epoch)

            # 保存acc
            acc1[epoch][client_id] = global_meter.test_meters[epoch][client_id].accuracy_score * 100
            precision[epoch][client_id] = global_meter.test_meters[epoch][client_id].precision_score
            recall[epoch][client_id] = global_meter.test_meters[epoch][client_id].recall_score
            ffff[epoch][client_id] =  2 * (precision[epoch][client_id] * recall[epoch][client_id]) / (precision[epoch][client_id] + recall[epoch][client_id])
            all_jieguo1['epoch' + str(epoch)]['client' + str(client_id)] = acc1[epoch][client_id]

            if client_id == args.num_client - 1:
                # all_jieguo1['epoch' + str(epoch)]['avg'] = (acc1[epoch][0] + acc1[epoch][1] + acc1[epoch][2] + acc1[epoch][3])/4.0
                all_jieguo1['epoch' + str(epoch)]['avg'] = (acc1[epoch][0] + acc1[epoch][1] + acc1[epoch][2])/3.0
            print("client{}当前准确率为：{}".format(client_id, acc1[epoch][client_id]))
            # 定义文件名
            filename = args.data_dir + '/test1_jieguo.txt'
            fpath = os.path.join(base_dir, filename)
            # 判断文件是否存在，如果存在则删除
            if os.path.exists(fpath):
                os.remove(fpath)
            with open(base_dir+ filename, 'w') as f:
                for key, value in all_jieguo1.items():
                    f.write(f'{key}: {value}\n')
            acc1.to_csv(base_dir+args.data_dir+'/test1_acc.csv', index=False)    
            precision.to_csv(base_dir+args.data_dir+'/test1_precision.csv', index=False)
            recall.to_csv(base_dir+args.data_dir+'/test1_recall.csv', index=False)
            ffff.to_csv(base_dir+args.data_dir+'/test1_f1.csv', index=False)
       

        global_meter.update_model(global_model,epoch,args.global_epoch)